# Rule-Based Features — Parameter Sensitivity Analysis

Sweeps the rule-based bias node parameter grid defined in `config.py`, computes the
Sortino ratio directly from feature signals (no model fitting needed), and runs
`generate_parameter_sensitivity_report()` to identify stable parameter regions.

**Workflow:**
1. Edit `config.py` (or override inline below) to set tickers, date range, bias_spec
2. Run all cells (Kernel → Restart & Run All)
3. Interact with the Plotly figures in Section 5

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import pandas as pd
from datetime import datetime
from dataclasses import replace

from feature_research.in_sample.rule_based.config import load_config
from feature_research.in_sample.rule_based.data_loader import (
    expand_bias_specs,
    load_features_for_combo,
    populate_cache_if_needed,
    param_combo_label,
)
from metrics.performance import SortinoRatio
from eda.parameter_analysis import generate_parameter_sensitivity_report
from utils.enums import Ticker, TimeFrame

print("Imports OK")

Imports OK


## 1. Configuration

Edit `config.py` for persistent changes, or use the override cell below for one-off runs.
`dataclasses.replace()` is used because `RuleBasedResearchConfig` is a frozen dataclass.

In [2]:
# --- Load defaults from config.py ---
config = load_config()

# ============================================================
# OVERRIDE CELL — edit here to customise without touching config.py
# ============================================================

# config = replace(config, tickers=[Ticker.ES])
# config = replace(config, start=datetime(2015, 1, 1), end=datetime(2024, 12, 31))
# config = replace(config,
#     bias_spec={
#         "module_name": "rsi_signal",
#         "timeframes": [TimeFrame.D],
#         "params": {
#             "rsi_period": [2, 3, 5, 7, 10, 14],
#             "oversold": 25.0,
#             "overbought": 65.0,
#             "strategy_mode": "long",
#             "exit_policy": "threshold_or_bars",
#             "exit_bars": 5,
#         },
#     }
# )

# --- Param sensitivity knobs ---
STABILITY_THRESHOLD = 0.8
TOP_K = 3
METRIC_COL = "sortino"
PLOT_3D_MODE = "surface_slices"  # options: "heatmap_slices", "surface_slices"

# ============================================================

# Derive varying param names (list-valued params only)
varying_params = [
    k for k, v in config.bias_spec["params"].items()
    if isinstance(v, list) and len(v) > 1
]
if not varying_params:
    print(
        "WARNING: No list-valued params found in bias_spec. "
        "Parameter sensitivity requires at least one param with multiple values. "
        "Results will be a single-point grid — sensitivity analysis is not meaningful."
    )
    varying_params = list(config.bias_spec["params"].keys())

print(f"Module   : {config.bias_spec['module_name']}")
print(f"Tickers  : {[t.name for t in config.tickers]}")
print(f"Period   : {config.start.date()} -> {config.end.date()}")
print(f"Varying  : {varying_params}")
print(f"METRIC: {METRIC_COL}  |  THRESHOLD: {STABILITY_THRESHOLD}")
print(f"3D MODE: {PLOT_3D_MODE}")


Module   : rsi_signal
Tickers  : ['ES', 'NQ', 'YM', 'RTY']
Period   : 2000-01-01 -> 2024-12-31
Varying  : ['rsi_period', 'oversold', 'overbought']
METRIC: sortino  |  THRESHOLD: 0.8
3D MODE: surface_slices


## 2. Feature Extraction

Loads feature + target data for every param combo. Uses the cache if `config.use_cache=True`.
Set `config.populate_cache=True` to auto-populate missing cache entries.

In [3]:
populate_cache_if_needed(config)

expanded_specs = expand_bias_specs(config.bias_spec)
print(f"Parameter grid: {len(expanded_specs)} combinations\n")

combo_data = {}
for spec in expanded_specs:
    result = load_features_for_combo(spec, config)
    if result is not None:
        feature, target, feature_col = result
        combo_data[param_combo_label(spec["params"])] = {
            "feature": feature,
            "target": target,
            "params": spec["params"],
        }
    else:
        print(f"  SKIP: {spec['params']}")

print(f"\nLoaded {len(combo_data)}/{len(expanded_specs)} combos successfully")

  Adding required auxiliary: atr (for log_return_atr/log_return_ewsd)
  Adding required auxiliary: ewsd (for log_return_atr/log_return_ewsd)
Populating cache for 568 combinations...
  Bias node specs: 142 (including required atr/ewsd if needed)
  Tickers: 4
  Date range: 2000-01-01 00:00:00 to 2024-12-31 00:00:00
  Max workers: 4
  [1/568] rsi_signal / NQ / D
  [2/568] rsi_signal / ES / D
  [3/568] rsi_signal / RTY / D
  [4/568] rsi_signal / YM / D
  [5/568] rsi_signal / ES / D
  [6/568] rsi_signal / NQ / D
  [7/568] rsi_signal / RTY / D
  [8/568] rsi_signal / YM / D
  [9/568] rsi_signal / NQ / D
  [10/568] rsi_signal / ES / D
  [11/568] rsi_signal / YM / D
  [12/568] rsi_signal / RTY / D
  [13/568] rsi_signal / NQ / D
  [14/568] rsi_signal / ES / D
  [15/568] rsi_signal / YM / D
  [16/568] rsi_signal / ES / D
  [17/568] rsi_signal / YM / D
  [18/568] rsi_signal / RTY / D
  [19/568] rsi_signal / NQ / D
  [20/568] rsi_signal / ES / D
  [21/568] rsi_signal / NQ / D
  [22/568] rsi_signal 

Partial cache coverage for rsisignal (ES, D). Cache starts at 2020-01-02 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.


  [499/568] rsi_signal / YM / D
  [500/568] rsi_signal / ES / D
  [501/568] rsi_signal / NQ / D
  [502/568] rsi_signal / RTY / D
  [503/568] rsi_signal / ES / D
  [504/568] rsi_signal / RTY / D
  [505/568] rsi_signal / YM / D
  [506/568] rsi_signal / NQ / D
  [507/568] rsi_signal / YM / D
  [508/568] rsi_signal / ES / D
  [509/568] rsi_signal / RTY / D
  [510/568] rsi_signal / NQ / D
  [511/568] rsi_signal / RTY / D
  [512/568] rsi_signal / ES / D
  [513/568] rsi_signal / YM / D
  [514/568] rsi_signal / NQ / D
  [515/568] rsi_signal / NQ / D
  [516/568] rsi_signal / RTY / D
  [517/568] rsi_signal / ES / D
  [518/568] rsi_signal / YM / D
  [519/568] rsi_signal / RTY / D
  [520/568] rsi_signal / YM / D
  [521/568] rsi_signal / ES / D
  [522/568] rsi_signal / NQ / D
  [523/568] rsi_signal / ES / D
  [524/568] rsi_signal / RTY / D
  [525/568] rsi_signal / YM / D
  [526/568] rsi_signal / NQ / D
  [527/568] rsi_signal / RTY / D
  [528/568] rsi_signal / YM / D
  [529/568] rsi_signal / NQ / D


Partial cache coverage for rsisignal (NQ, D). Cache starts at 2020-01-02 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.
Partial cache coverage for rsisignal (YM, D). Cache starts at 2002-04-05 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.
Partial cache coverage for rsisignal (RTY, D). Cache starts at 2017-06-02 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.
Partial cache coverage for atr (ES, D). Cache starts at 2000-01-03 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.
Partial cache coverage for atr (NQ, D). Cache starts at 2000-01-03 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.
Partial cache coverage for atr (YM, D). Cache starts at 2002-04-05 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.
Partial cache coverage for atr (RTY, D). Cache starts at 2017-06-02 00:00:00, requested 2000-01-01 00:00:00. Proceeding with available data.



Loaded 140/140 combos successfully


## 3. Metric Grid

Rule-based features are binary signals (1 = trade, 0 = no trade).
The Sortino ratio is computed directly on returns where the signal fires — no model fitting required.

In [4]:
metric = SortinoRatio(annualization_factor=252)
rows = []

for spec in expanded_specs:
    label = param_combo_label(spec["params"])
    if label not in combo_data:
        continue

    data = combo_data[label]
    feature = data["feature"]
    target = data["target"]

    try:
        # Rule-based: feature IS the signal — no model fitting
        signals = feature.values == 1
        selected_returns = target.values[signals]

        if len(selected_returns) < 5:
            print(f"  SKIP [{label}]: too few signals ({len(selected_returns)}) to compute reliable metric")
            continue

        metric_value = metric.compute(selected_returns)

        row = {
            f"param{k+1}_value": spec["params"][key]
            for k, key in enumerate(varying_params)
        }
        row[METRIC_COL] = metric_value
        rows.append(row)

    except Exception as e:
        print(f"  ERROR [{label}]: {e}")

results_df = pd.DataFrame(rows)
print(f"Grid results: {len(results_df)} rows")
results_df

  SKIP [exit_bars_5__exit_policy_threshold_or_bars__overbought_95__oversold_5__rsi_period_7__strategy_mode_long]: too few signals (0) to compute reliable metric
  SKIP [exit_bars_5__exit_policy_threshold_or_bars__overbought_90__oversold_5__rsi_period_7__strategy_mode_long]: too few signals (0) to compute reliable metric
  SKIP [exit_bars_5__exit_policy_threshold_or_bars__overbought_85__oversold_5__rsi_period_7__strategy_mode_long]: too few signals (0) to compute reliable metric
  SKIP [exit_bars_5__exit_policy_threshold_or_bars__overbought_80__oversold_5__rsi_period_7__strategy_mode_long]: too few signals (0) to compute reliable metric
  SKIP [exit_bars_5__exit_policy_threshold_or_bars__overbought_75__oversold_5__rsi_period_7__strategy_mode_long]: too few signals (0) to compute reliable metric
  SKIP [exit_bars_5__exit_policy_threshold_or_bars__overbought_70__oversold_5__rsi_period_7__strategy_mode_long]: too few signals (0) to compute reliable metric
  SKIP [exit_bars_5__exit_policy_t

,param1_value,param2_value,param3_value,sortino
0,2,5,95,0.234274
1,2,5,90,0.205962
2,2,5,85,0.114821
3,2,5,80,0.161934
4,2,5,75,0.089765
...,...,...,...,...
128,7,25,85,1.224437
129,7,25,80,1.224437
130,7,25,75,1.224437
131,7,25,70,1.224437


## 4. Parameter Sensitivity Report

Runs: neighbor smoothing → stable region detection → recommendations.

In [5]:
if results_df.empty:
    raise RuntimeError(
        f"No results collected. Check that combos loaded: "
        f"{len(combo_data)}/{len(expanded_specs)} loaded. "
        f"Ensure cache is available or set config.populate_cache=True."
    )

report = generate_parameter_sensitivity_report(
    results_df=results_df,
    param_names=varying_params,
    metric_col=METRIC_COL,
    stability_threshold=STABILITY_THRESHOLD,
    top_k=TOP_K,
    plot_3d_mode=PLOT_3D_MODE,
)

print(f"Module              : {config.bias_spec['module_name']}")
print(f"Metric              : {report.metric_name}")
print(f"Grid size           : {report.n_parameter_combinations}")
print(f"Stable regions      : {report.n_stable_regions}")
print(f"Mean stability ratio: {report.mean_stability_ratio:.3f}")
print(f"% stable combos     : {report.pct_stable_combinations:.1%}")
print(f"\nTop-{TOP_K} recommendations:")
for i, combo in enumerate(report.top_k_combinations, 1):
    params_str = ", ".join(
        f"{name}={val}" for name, val in zip(varying_params, combo)
    )
    print(f"  {i}. {params_str}")


Module              : rsi_signal
Metric              : sortino
Grid size           : 133
Stable regions      : 1
Mean stability ratio: -0.296
% stable combos     : 72.9%

Top-3 recommendations:
  1. rsi_period=5, oversold=10, overbought=95
  2. rsi_period=5, oversold=10, overbought=65
  3. rsi_period=5, oversold=10, overbought=90


## 5. Interactive Visualizations

The appropriate plot is selected automatically based on grid dimensionality:
- 1D grid → raw + smoothed line with shaded stable regions
- 2D grid → heatmap with stability contour overlay
- 3D+ grid → heatmap slices with dropdown + slider controls

In [6]:
if report.plot_1d is not None:
    report.plot_1d.show()

if report.plot_2d is not None:
    report.plot_2d.show()

if report.plot_3d is not None:
    report.plot_3d.show()

## 6. Optional Save

Uncomment cells below to save plots as interactive HTML or grid results as parquet.

In [7]:
# --- OPTIONAL: Save outputs (uncomment to enable) ---

# from pathlib import Path
# save_dir = config.reports_dir / "param_sensitivity"
# save_dir.mkdir(parents=True, exist_ok=True)

# if report.plot_1d is not None:
#     report.plot_1d.write_html(save_dir / f"param_sensitivity_1d_{config.bias_spec['module_name']}.html")
#     print(f"Saved 1D plot -> {save_dir}")

# if report.plot_2d is not None:
#     report.plot_2d.write_html(save_dir / f"param_sensitivity_2d_{config.bias_spec['module_name']}.html")
#     print(f"Saved 2D plot -> {save_dir}")

# if report.plot_3d is not None:
#     report.plot_3d.write_html(save_dir / f"param_sensitivity_3d_{config.bias_spec['module_name']}.html")
#     print(f"Saved 3D plot -> {save_dir}")

# report.grid_results.to_parquet(save_dir / "param_sensitivity_grid.parquet")
# print(f"Saved grid results -> {save_dir / 'param_sensitivity_grid.parquet'}")